# Task 2.1: Feature Engineering for Rainfall Prediction
## Singapore Rainfall, 2017–2024

This notebook prepares the cleaned rainfall data for predicting whether a monitoring station will accumulate more than a specified rainfall amount over the next 30 minutes. Building on the findings from Task 1, PySpark is used to construct features describing station location, calendar patterns, and recent rainfall conditions.

The preparation covers:

- **Feature construction:** Create 12 model features, including coordinates, hour, month, current and lagged rainfall, recent rainfall sums, wet-interval counts, and the requested threshold.
- **Target construction:** Sum the next 6 five-minute readings and assign labels indicating whether the total exceeds thresholds of **0, 1, 2.5, 5, and 10 mm**.
- **Data-quality checks:** Retain observations with complete historical and future windows, correctly spaced readings, and no missing required values.
- **Chronological splitting:** Use **2017–2022 for training, 2023 for validation, and 2024 for testing**, excluding target windows that cross split boundaries.
- **Sampling and class checks:** Sample eligible observations to manage computation, expand each into 5 threshold-specific examples, and inspect positive and negative label counts.

Rainfall features use only current and earlier readings, while future rainfall is used exclusively to construct labels. The resulting datasets support hyperparameter tuning and final model evaluation in the subsequent notebooks.

## 2.1.1 Spark Session

In [1]:
import socket
from functools import reduce

from pyspark.sql import SparkSession, Window, DataFrame

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pyspark.sql.functions as F

import time
from pyspark import StorageLevel

In [2]:
spark = SparkSession.builder.getOrCreate()

spark.conf.set("spark.sql.session.timeZone", "Asia/Singapore")
spark.conf.set("spark.sql.shuffle.partitions", "128")
spark.conf.set("spark.sql.adaptive.enabled", "true")

print("Running on:   ", socket.gethostname())
print("Spark master: ", spark.sparkContext.master)
print("Spark version:", spark.version)

Running on:    rainfall-dev-m
Spark master:  yarn
Spark version: 3.5.3


In [3]:
# Rainfall Multiplier per Day there are 12 five-minute intervals per hour and 24 hours per day

RF_MULTIPLIER = (24 * 60) / 5

## 2.1.2 Reading the Cleaned Rainfall Data

### Features

The following features are loaded directly from the cleaned dataset:

- **`reading_value`:** Five-minute rainfall amount recorded at time t.
- **`location_longitude`:** Station longitude.
- **`location_latitude`:** Station latitude.

In [4]:
rain = spark.read.parquet("gs://dsa5208-rainfall-sixing-data/parquet/")

## 2.1.3 Auto-Correlation of Rainfall and Its Timelags 

In [6]:
def lookback(seconds):
    return (Window.partitionBy("station_id")
            .orderBy("ts_long")
            .rangeBetween(-seconds, 0))

### Features

- **`lag_5m`:** Rainfall reading recorded 5 minutes before t.
- **`lag_30m`:** Rainfall reading recorded 30 minutes before t.
- **`lag_1h`:** Rainfall reading recorded 60 minutes before t.

The loop below creates these features separately for each station and checks that the time gaps match the required durations. Values with incorrect time gaps are set to null.

In [7]:
rain_lag = rain.withColumn("ts_long", F.col("timestamp").cast("long"))

w = Window.partitionBy("station_id").orderBy("ts_long")

lags = {"lag_5m": 1, "lag_30m": 6, "lag_1h": 12}

for name, k in lags.items():
    rain_lag = (rain_lag
        .withColumn(name, F.lag("reading_value", k).over(w))
        .withColumn(name + "_ts", F.lag("ts_long", k).over(w)))
    
    # null out where the gap isn't the expected duration
    rain_lag = rain_lag.withColumn(name,
        F.when(F.col("ts_long") - F.col(name + "_ts") == k * 300,
               F.col(name)))
    rain_lag = rain_lag.drop(name + "_ts")

### Features

- **`rain_cumsum_30m`:** Sum of 6 five-minute rainfall readings from t − 25 minutes through t.
- **`rain_cumsum_1h`:** Sum of 12 five-minute rainfall readings from t − 55 minutes through t.
- **`wet_count_30m`:** Number of positive rainfall readings among the 6 readings from t − 25 minutes through t. Range: 0–6.
- **`hour`:** Hour extracted from the observation timestamp in Singapore time.
- **`month`:** Calendar month extracted from the observation timestamp.

The code below calculates these features separately for each station. It also creates `future_30m` for target construction, extracts `year` for chronological splitting, and calculates reading counts and gap flags used to filter incomplete windows in the next step.

In [8]:
past_30m = lookback(1500)     # Six readings: t−25 through t.
past_1h = lookback(3300)      # Twelve readings: t−55 through t.
history_1h = lookback(3600)   # Thirteen readings, including t−60 for lag_1h.

future_window = (
    Window.partitionBy("station_id")
          .orderBy("ts_long")
          .rangeBetween(1, 1800)
)

# Flag gaps that are not exactly five minutes.
rain_lag = (
    rain_lag
    .withColumn("_previous_ts", F.lag("ts_long").over(w))
    .withColumn(
        "_bad_gap",
        F.when(
            F.col("ts_long") - F.col("_previous_ts") == 300,
            0
        ).otherwise(1)
    )
)

rain_lag = (
    rain_lag

    # Historical rainfall features.
    .withColumn(
        "rain_cumsum_30m",
        F.sum("reading_value").over(past_30m)
    )
    .withColumn(
        "rain_cumsum_1h",
        F.sum("reading_value").over(past_1h)
    )
    .withColumn(
        "wet_count_30m",
        F.sum(
            (F.col("reading_value") > 0).cast("long")
        ).over(past_30m)
    )

    # Check the thirteen historical readings and twelve connecting gaps.
    .withColumn(
        "_history_count",
        F.count("reading_value").over(history_1h)
    )
    .withColumn(
        "_history_bad_gap",
        F.max("_bad_gap").over(past_1h)
    )

    # Check the six future readings and their connecting gaps.
    .withColumn(
        "_future_count",
        F.count("reading_value").over(future_window)
    )
    .withColumn(
        "_future_bad_gap",
        F.max("_bad_gap").over(future_window)
    )
    .withColumn(
        "future_30m",
        F.sum("reading_value").over(future_window)
    )

    # Use the observation timestamp consistently.
    .withColumn("hour", F.hour("timestamp"))
    .withColumn("month", F.month("timestamp"))
    .withColumn("year", F.year("timestamp"))
)

In [9]:
feature_cols = [
    "location_latitude",
    "location_longitude",
    "hour",
    "month",
    "reading_value",
    "lag_5m",
    "lag_30m",
    "lag_1h",
    "rain_cumsum_30m",
    "rain_cumsum_1h",
    "wet_count_30m",
    "thr"
]

THRESHOLDS = [0.0, 1.0, 2.5, 5.0, 10.0]
past_feature_cols = [c for c in feature_cols if c != "thr"]

base = (
    rain_lag
    .filter(
        (F.col("_history_count") == 13) &
        (F.col("_history_bad_gap") == 0) &
        (F.col("_future_count") == 6) &
        (F.col("_future_bad_gap") == 0)
    )
    .filter(F.col("year").between(2017, 2024))
    .select(
        "station_id", "timestamp", "year",
        *past_feature_cols,
        "future_30m"
    )
    .dropna(subset=past_feature_cols + ["future_30m"])

    # Store on executor disks instead of trying to hold everything in memory.
    .persist(StorageLevel.DISK_ONLY)
)

# Complete feature preparation now, so it is not included in model-fit timing.
# This first full-data preparation step can still take time.
started = time.perf_counter()

print("Eligible station-time rows:", base.count())
print(
    "Feature preparation seconds:",
    round(time.perf_counter() - started, 1)
)

target_end = F.col("timestamp") + F.expr("INTERVAL 30 MINUTES")

train = base.filter(
    (F.col("year") <= 2022) &
    (target_end < F.lit("2023-01-01").cast("timestamp"))
)

val = base.filter(
    (F.col("year") == 2023) &
    (target_end < F.lit("2024-01-01").cast("timestamp"))
)

test = base.filter(
    (F.col("year") == 2024) &
    (target_end < F.lit("2025-01-01").cast("timestamp"))
)

26/10/16 14:23:03 WARN SimpleTableFunctionRegistry: The function read_files replaced a previously registered function.


Eligible station-time rows: 47000882
Feature preparation seconds: 74.6


### Sampling and Label Construction

The code below prepares smaller datasets for model training and evaluation:

1. **Sample observations:** Randomly sample approximately 0.5% of training, 5% of validation, and 10% of test observations without replacement, using seed 42. Sampling occurs after feature and future-total construction and does not depend on the label.
2. **Add rainfall thresholds:** Expand each sampled station–time observation into 5 examples, using thresholds of 0, 1, 2.5, 5, and 10 mm.
3. **Create labels:** Assign label 1 when `future_30m` is strictly greater than `thr`; otherwise, assign label 0.
4. **Prepare for modeling:** Retain the model features and label, repartition each dataset into 32 partitions, and cache it using memory and disk.
5. **Check class representation:** Report positive and negative counts and positive-label percentages for each threshold. Stop if a sample is empty or any threshold has only 1 class.

Sampling reduces computational cost while giving all eligible observations the same selection probability within each dataset. Having both classes present is a minimum check; it does not guarantee enough positive examples for reliable modeling.

In [ ]:
# 1. Takes a small uniform sample to reduce training time.
# 2. Gives positive and negative examples the same selection chance.
# 3. Prints their counts separately for every rainfall threshold.

SEED = 42

# These are starting fractions for checking that training runs successfully.
# Increase them later if too few positive examples are retained.
TRAIN_FRACTION = 0.005       # Approximately 0.5% of training station-times.
VALIDATION_FRACTION = 0.05   # Approximately 5% of validation station-times.
TEST_FRACTION = 0.10         # Approximately 10% of test station-times. 

ML_PARTITIONS = 32


def prepare_sample(df, fraction):

    # STEP 1: Randomly select station-time observations.
    # Selection does not depend on whether rainfall exceeds a threshold.
    #
    # Historical features and future rainfall were already calculated.
    # Do not sample raw readings before calculating those windows.
    sampled = df.sample(
        withReplacement=False,
        fraction=fraction,
        seed=SEED
    )

    # STEP 2: Ask all five threshold questions for each selected observation.
    #
    # Example: if future_30m = 3 mm:
    # thr = 1 mm -> label 1
    # thr = 5 mm -> label 0
    labelled = (
        sampled
        .withColumn(
            "thr",
            F.explode(
                F.array(*[F.lit(float(t)) for t in THRESHOLDS])
            )
        )
        .withColumn(
            "label",
            (F.col("future_30m") > F.col("thr")).cast("double")
        )
    )

    # STEP 3: Keep only model inputs and the label.
    # Cache this smaller dataset so repeated training actions can reuse it.
    return (
        labelled
        .select(*feature_cols, "label")
        .repartition(ML_PARTITIONS)
        .persist(StorageLevel.MEMORY_AND_DISK)
    )


train_final = prepare_sample(train, TRAIN_FRACTION)
val_final = prepare_sample(val, VALIDATION_FRACTION)
test_final = prepare_sample(test, TEST_FRACTION)


# STEP 4: Inspect imbalance separately for each threshold.
# A higher rainfall threshold may have fewer positive examples.
for dataset_name, dataset in [
    ("Training", train_final),
    ("Validation", val_final),
    ("Test", test_final),
]:
    summary = (
        dataset
        .groupBy("thr")
        .agg(
            F.count("*").alias("total"),
            F.sum("label").alias("positives")
        )
        .orderBy("thr")
        .collect()
    )

    # This action also fills the cache before model training starts.
    print(f"\n{dataset_name} class distribution")

    if not summary:
        raise ValueError(f"{dataset_name} sample is empty.")

    for row in summary:
        total = row["total"]
        positives = int(row["positives"])
        negatives = total - positives
        positive_pct = 100 * positives / total

        print(
            f"> {row['thr']:g} mm: "
            f"label 1 = {positives:,}, "
            f"label 0 = {negatives:,}, "
            f"positive percentage = {positive_pct:.2f}%"
        )

        # Stop if a threshold has no examples of one class.
        # Increase the relevant sample fraction and rerun this cell.
        if positives == 0 or negatives == 0:
            raise ValueError(
                f"{dataset_name}, threshold {row['thr']:g} mm: "
                "both classes are required. Increase the sample fraction."
            )



26/10/16 14:32:24 WARN CacheManager: Asked to cache already cached data.
26/10/16 14:32:24 WARN CacheManager: Asked to cache already cached data.
26/10/16 14:32:24 WARN CacheManager: Asked to cache already cached data.



Training class distribution
> 0 mm: label 1 = 10,283, label 0 = 159,366, positive percentage = 6.06%
> 1 mm: label 1 = 4,083, label 0 = 165,566, positive percentage = 2.41%
> 2.5 mm: label 1 = 2,403, label 0 = 167,246, positive percentage = 1.42%
> 5 mm: label 1 = 1,459, label 0 = 168,190, positive percentage = 0.86%
> 10 mm: label 1 = 690, label 0 = 168,959, positive percentage = 0.41%

Validation class distribution
> 0 mm: label 1 = 22,306, label 0 = 311,287, positive percentage = 6.69%
> 1 mm: label 1 = 8,860, label 0 = 324,733, positive percentage = 2.66%
> 2.5 mm: label 1 = 5,148, label 0 = 328,445, positive percentage = 1.54%
> 5 mm: label 1 = 3,013, label 0 = 330,580, positive percentage = 0.90%
> 10 mm: label 1 = 1,390, label 0 = 332,203, positive percentage = 0.42%

Test class distribution
> 0 mm: label 1 = 40,305, label 0 = 593,159, positive percentage = 6.36%
> 1 mm: label 1 = 16,260, label 0 = 617,204, positive percentage = 2.57%
> 2.5 mm: label 1 = 9,332, label 0 = 624,13

## 2.1.4 Output the Datasets to Parquet

In [16]:
#train_final.write.mode("overwrite").parquet("gs://dsa5208-rainfall-sixing-data/train_parquet/")
#val_final.write.mode("overwrite").parquet("gs://dsa5208-rainfall-sixing-data/val_parquet/")
#test_final.write.mode("overwrite").parquet("gs://dsa5208-rainfall-sixing-data/test_parquet/")
#test.write.mode("overwrite").parquet("gs://dsa5208-rainfall-sixing-data/full_test_parquet/")

26/10/16 14:38:40 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /hadoop/spark/tmp/spark-9ca73388-7314-4bbd-b1c3-e2f5b3de5f60/pyspark-0edb48fc-df52-4c03-a27a-38b9e299411e. Falling back to Java IO way
java.io.IOException: Failed to delete: /hadoop/spark/tmp/spark-9ca73388-7314-4bbd-b1c3-e2f5b3de5f60/pyspark-0edb48fc-df52-4c03-a27a-38b9e299411e
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:174) ~[spark-common-utils_2.12-3.5.3.jar:3.5.3]
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:109) ~[spark-common-utils_2.12-3.5.3.jar:3.5.3]
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:90) ~[spark-common-utils_2.12-3.5.3.jar:3.5.3]
	at org.apache.spark.util.SparkFileUtils.deleteRecursively(SparkFileUtils.scala:121) ~[spark-common-utils_2.12-3.5.3.jar:3.5.3]
	at org.apache.spark.util.SparkFileUtils.deleteRecursively$(SparkFileUtils.scala:120) ~[spark-common-uti